In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
#!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [2]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [3]:
print("length of dataset in characters:", len(text))

length of dataset in characters: 1115394


In [4]:
print(text[:50])

First Citizen:
Before we proceed any further, hear


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(''.join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


Since the goal is to build a "character" based Language model, the smallest unit of for tokens will be characters.

# Tokenization and Training-Val split

In [6]:
# Create mapping from characters to integers
stoi = {ch:i for i, ch, in enumerate(chars)} # string to integers. Produces {'\n': 0, ' ': 1, '!': 2,...
itos = {i:ch for i, ch in enumerate(chars)} # in reverse: {0: '\n',  1: ' ',  2: '!',...

encode = lambda s: [stoi[c] for c in s] # encoder: takes a string and return list of integers
decode = lambda i: "".join([itos[n] for n in i]) # decoder: takes a list of integers and returns string

print(encode("putos "))
print(decode(encode("putos ")))

[54, 59, 58, 53, 57, 1]
putos 


Now we can encode all the data

In [7]:
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


Splitting data into training and validation

In [8]:

n = int(len(data)*0.9) # First 90% will be training
train_data = data[:n]
val_data = data[n:]

# Data Loader and batches of chunks

### Things to consider.
We do not need a prediction success of 100% since we do not want to predict exactly the Shakespear text, but to generate Shakespear-like text.

Also when training a Transformer we do not use all text at once, because it is computationall expensive (given the nature of the transformer). So what is usually done is to extract random chunks of text from the training set (specific lenght, or maximum lenght; this is the block size, or context length).

In [9]:
block_size = 8
train_data[:block_size+1] # Notice the plus one

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

The magic of training is that given a single block of training we have multiple training units.

For example. If we only take token 18 in the context window, the goal (target) is to predict 47. Example 1.

If we take 18 and 47 in the context window, the goal is to predict 56; this is the second example.

And so on and so forth we can take 18 to 47 (8 numbers in total; the size of the context) to predict 58. the eight example from a single training block.

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    targets = y[t]
    print(f"When the context is {context}, the target: {target}")

When the context is tensor([18]), the target: 47
When the context is tensor([18, 47]), the target: 56
When the context is tensor([18, 47, 56]), the target: 57
When the context is tensor([18, 47, 56, 57]), the target: 58
When the context is tensor([18, 47, 56, 57, 58]), the target: 1
When the context is tensor([18, 47, 56, 57, 58,  1]), the target: 15
When the context is tensor([18, 47, 56, 57, 58,  1, 15]), the target: 47
When the context is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target: 58


Having multiple examples from a single training block is not just for computational efficiency, but also because in inference we want the transformer to be able to predict having as little as "1 token" in the context, up to "8 tokens". 

So this answer one queestion I had in mind earlier. I was thinking that this way of training was just predicting the next token from the "single" previous one. That sounded not logical, but now I understood that is not the case.

There is an additional dimension (other than block size and ...) that we care about: __Batch dimension.__

For every training step (feeding/inputing data to the tensor, every forward/backward pass) we not use a single training block, but multiple; and this number is given by the batch size. Multiple training blocks or "a batch" is encapsulated into "a" tensor. This is parallel processing (one for each GPU).

In [11]:
torch.manual_seed(1337)

batch_size = 4 # how many "independent" sequences will be process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # Choosing 4 (batch_size) random position between all tokens (of the training or val dataset)
    # e.g. tensor([1078327,  453969,   41646,  671252])
    
    # Defining the "batch" (set of blocks) as a tensor.
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# Example of batch
xb, yb = get_batch("train")
print("input:")
print(xb.shape)
print(xb)
print("target:")
print(yb.shape)
print(yb)

print("---")

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension. _____¿Why is it called t dimension?_____
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"When input is {context} the target: {target}")


input:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
target:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
---
When input is tensor([24]) the target: 43
When input is tensor([24, 43]) the target: 58
When input is tensor([24, 43, 58]) the target: 5
When input is tensor([24, 43, 58,  5]) the target: 57
When input is tensor([24, 43, 58,  5, 57]) the target: 1
When input is tensor([24, 43, 58,  5, 57,  1]) the target: 46
When input is tensor([24, 43, 58,  5, 57,  1, 46]) the target: 43
When input is tensor([24, 43, 58,  5, 57,  1, 46, 43]) the target: 39
When input is tensor([44]) the target: 53
When input is tensor([44, 53]) the target: 56
When input is tensor([44, 53, 56]) the target: 1
When input is tensor([

Using n-gram (bigram in this case) prediction model, because is the simplest token prediction model (predict the next from the last element)

In [12]:
xb

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

In [16]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__() ## Code to call (class) constructor of the parent class (nn.Module)
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # Creates a lookup table (matrix) of size vocab_size × vocab_size
        # I will call it Logit Table
         ## The logits are the "score" (that will be normalized to probabilities) for each 
         ## token in the token dictionary in order to "predict" the next token. 
        # THIS TABLE is the thing that the training will improve. For each word (row) we have the probability/logits of each other 
        # token (columns). The context will always be only "1" token to predict the next one.

        # At the beginning we just initialize this table.

    def forward(self, idx, targets=None):

        # idx and targets are both (B, T) tensor of integers.
         ## B: batch dim. T: time dim.
         ## idx: is a tensor of all the IDs of the input tokens
        logits = self.token_embedding_table(idx) # (B,T,C)
        ## So this code is ask the Logits Table for the logits of the inputs (the tokens being process), 
        # there are multiple inputs, the total given by the number of batches and "times" (T: length of sequence)

        
        
        # Now that we make predictions (using the logits) we can measure loss function 
        # (or quality of prediction) and a good way to do that is using the
        # "negative log likelihood loss". In pytorch is given by:
        
        if targets == None:
            loss = None
        else:
            ## Reshaping logits to fit into pytorch standard (B,C,T)
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # This takes all inputs (e.g. look at xb) and 
                                        # stretch them, convert them to a single dimension: a vector (line)
                                        # But preserving the channel (C) dim as the 2nd dim.
            targets = targets.view(B*T) # or for Pytorch to guess: targets.view(-1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    # Now that we can measure the quality of the model on some data to "generate" from the model.
    # The details of this are in previous videos of Andrej
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx) # this is like "m(xb, yb)" invokes the model instance. # We don't use the loss here
            # focus only on the last time step (i.e. last t of T)
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1) ## This selects "randomly" a token from the vocab given 
                                                                        ## the weights from the probabilities (i.e. probs, in this example)
                                                                        # This is the prediction of the next token for each batch.
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb) # This outputs the (loss and) logits  for every input (total inputs given by B times T)
print(logits.shape)
print(loss)
idx = torch.zeros((1,1), dtype=torch.long) # Time and Batch will be 1, i.e. (1,1) tensor with value 0. Remember 0 stands for "\n" in our vocab embedding
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))



torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


This Bigram model implies that the tokens within the context doe not talk to each other. Because each input token makes prediction solely on itself.

## Training the model

In [17]:
# Creating PyTorch optimizer object
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [25]:
batch_size = 32
for steps in range(10_000):
    # sample batch of data
    xb, yb, = get_batch("train")

    # evaluate loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.4693028926849365


In [29]:
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


Wint pas IUComety rtonossoit
To,
An dsuneroncu ren hinder's,
sapp, her waige het, andanseve otoe, ngr are? tanol hat'seo Obere tiredsthorithly t ishe.
FRonis l omyo acem y r, I matolis s ay then desous s crs!

RWhathin ceithin$EDI inresvenou g BEYory alow lelld,
ANGR:

Why heas hast tthyonat,


LINTive.

Thart,
LAUM:
Wh tt than reinghe ithatod ht, athest berth. ishountopo min pean aw thend res! thathert as hinory t forde E hanghild my,
Wou ICHanthidon,
IA:
RDUS kere senss, fanowila's:
K: a whe i


In [30]:
import torch

# Check if Apple's Metal backend is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using M1 GPU!")
else:
    device = torch.device("cpu")

Using M1 GPU!


# The mathematical trick in self-attention

In [34]:
# consider the following toy example:

torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time, channels
x = torch.rand(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# We want x[b,t] = mean_[i<=t] x[b,i]
xbow = torch.zeros((B,T,C)) # bow: bag of words
for b in range(B):
    for t in range(T):
        xprev = x[b :t+1] # (t, C)
        xbow[b,t] = torch.mean(xprev, 0)